In [0]:
from pyspark.sql.functions import current_timestamp

# 1. Ler os dados que você acabou de carregar no catálogo
df_origem = spark.read.table("workspace.default.olist_orders_dataset")

# 2. Adicionar metadado (data e hora em que o dado foi processado)
df_bronze = df_origem.withColumn("data_ingestao", current_timestamp())

# 3. Criar a Camada Bronze salvando como uma nova tabela Delta
df_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_olist_pedidos")

print("Camada Bronze criada com sucesso!")


Camada Bronze criada com sucesso!


In [0]:
# 1. Ler os dados diretamente da Camada Bronze que acabamos de salvar
df_bronze_pedidos = spark.read.table("workspace.default.bronze_olist_pedidos")

# 2. Limpeza: Remover registros onde o ID do pedido esteja nulo ou vazio
df_silver_pedidos = df_bronze_pedidos.filter(df_bronze_pedidos.order_id.isNotNull())

# 3. Remover possíveis linhas duplicadas para garantir a qualidade dos dados
df_silver_pedidos = df_silver_pedidos.dropDuplicates(["order_id"])

# 4. Salvar o resultado limpo na Camada Silver
df_silver_pedidos.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_olist_pedidos")

print("Camada Silver criada com sucesso!")


Camada Silver criada com sucesso!


In [0]:
from pyspark.sql.functions import col, sum, count

# 1. Ler os dados limpos diretamente da Camada Silver
df_silver = spark.read.table("workspace.default.silver_olist_pedidos")

# 2. Agrupar por ID do produto para calcular faturamento e quantidade vendida
# Como a tabela inicial continha colunas de itens, faremos a conta direto nela
df_gold = df_silver.groupBy("product_id") \
                   .agg(sum(col("price").cast("double")).alias("faturamento_total"), 
                        count("order_id").alias("quantidade_vendida")) \
                   .orderBy(col("faturamento_total").desc()) # Ordena do mais vendido para o menos vendido

# 3. Salvar o resultado final agregado na Camada Gold
df_gold.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_produtos_mais_vendidos")

print("Camada Gold criada com sucesso! Veja abaixo o resultado:")

# 4. Mostrar na tela as top 10 linhas para conferirmos o resultado
display(df_gold.limit(10))


Camada Gold criada com sucesso! Veja abaixo o resultado:


product_id,faturamento_total,quantidade_vendida
null,null,99441


In [0]:
from pyspark.sql.functions import col, sum, count

# 1. Carregar a tabela original que você criou no início (ela tem os IDs dos produtos e preços de verdade)
df_itens_vendas = spark.read.table("workspace.default.olist_orders_dataset")

# 2. Filtrar os nulos para garantir a limpeza na camada Silver corrigida
df_itens_limpos = df_itens_vendas.filter(col("product_id").isNotNull() & col("price").isNotNull())

# 3. Criar a agregação Gold com os dados reais de faturamento
df_gold_correto = df_itens_limpos.groupBy("product_id") \
                                 .agg(sum(col("price").cast("double")).alias("faturamento_total"), 
                                      count("order_id").alias("quantidade_vendida")) \
                                 .orderBy(col("faturamento_total").desc())

# 4. Sobrescrever a tabela Gold com os dados corretos
df_gold_correto.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_produtos_mais_vendidos")

print("Pipeline corrigido com sucesso! Veja os top 10 produtos reais abaixo:")
display(df_gold_correto.limit(10))


Pipeline corrigido com sucesso! Veja os top 10 produtos reais abaixo:


product_id,faturamento_total,quantidade_vendida
bb50f2e236e5eea0100680137654686c,63885.0,195
6cdd53843498f92890544667809f1595,54730.200000000106,156
d6160fb7873f184099d9bc95e30376af,48899.34,35
d1c427060a0f73f6b889a5c7c61f2ac4,47214.50999999998,343
99a4788cb24856965c36a24e339b6058,43025.56000000037,488
3dd2a17168ec895c781a9191c1e95ad7,41082.60000000021,274
25c38557cf793876c5abdd5931f922db,38907.32000000001,38
5f504b3a1c75b73d6151be81eb05bdc9,37733.90000000001,63
53b36df67ebb7c41585e8d54d6772e08,37683.42000000013,323
aca2eb7d00ea1a7b8ebd4e68314663af,37608.900000000314,527


Databricks visualization. Run in Databricks to view.